# 03 — Model: two-tower TruncatedSVD + content TF-IDF

**Project:** Micro-Cert Recommender (H07)

Collaborative tower: TruncatedSVD on the implicit interaction matrix produces `U` (learner factors) and `V` (cert factors). Content tower: TF-IDF over `skills_taught`. Both serialise to `models/two_tower.joblib` and are loaded once at FastAPI startup.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from microcert_rec.data import PROCESSED, load_all, make_training_artifacts
from microcert_rec import models
from microcert_rec.models import fit, save
from microcert_rec.features import build_interaction_matrix, fit_cert_tfidf, learner_text

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'learners.parquet').exists():
    learners, certs, interactions = load_all()
else:
    learners, certs, interactions = make_training_artifacts()

## 1. Fit the two-tower bundle

In [ ]:
art = fit(learners, certs, interactions, k=32)
print('U shape:', art['U'].shape)
print('V shape:', art['V'].shape)
print('content matrix shape:', art['X_certs'].shape)

## 2. SVD scree

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(np.cumsum(art['svd'].explained_variance_ratio_), marker='o', color='#3a7ca5')
ax.set_xlabel('factor index'); ax.set_ylabel('cumulative explained variance')
ax.set_title('Collaborative-tower SVD scree')
plt.tight_layout(); plt.show()

## 3. Score one learner via the collaborative tower

In [ ]:
lid = learners.iloc[0]['learner_id']
i = art['learner_ids'].index(lid)
u = art['U'][i]
cf_score = art['V'] @ u
cf_score = (cf_score - cf_score.min()) / (cf_score.max() - cf_score.min() + 1e-9)
top_cf = np.argsort(-cf_score)[:8]
certs.iloc[top_cf][['cert_id', 'title', 'theme']].assign(cf=cf_score[top_cf].round(3))

## 4. Score the same learner via the content tower

In [ ]:
skill_cols = [c for c in learners.columns if c.startswith('skill__')]
skills = [c.replace('skill__', '') for c in skill_cols if learners.iloc[0][c] == 1]
qv = art['tfidf'].transform([learner_text(skills)]).toarray()
ct_score = cosine_similarity(qv, art['X_certs']).ravel()
top_ct = np.argsort(-ct_score)[:8]
certs.iloc[top_ct][['cert_id', 'title', 'theme']].assign(content=ct_score[top_ct].round(3))

## 5. Two-tower combined score (mean blend)

In [ ]:
blend = 0.5 * cf_score + 0.5 * ct_score
top = np.argsort(-blend)[:10]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(certs.iloc[top]['title'][::-1], blend[top][::-1], color='#4a7c59')
ax.set_xlabel('combined score')
ax.set_title(f'Two-tower top-10 for learner {lid}')
plt.tight_layout(); plt.show()

## 6. Theme distribution of top-10 — sanity check

If a learner's primary theme is 'Data', their top-10 should over-index on Data certs.

In [ ]:
primary = learners.iloc[0]['primary_theme']
themes = certs.iloc[top]['theme'].value_counts(normalize=True)
print(f'primary theme: {primary}')
themes

## 7. Persist the bundle

In [ ]:
path = save(art, 'two_tower.joblib')
print('saved ->', path)

## 8. Hand-off to `04_eval.ipynb`

The persisted bundle is now the production artefact. Evaluation measures recall@k under leave-one-out replay and quantifies the cold-start lift from the content tower.